# Fertilizer Prediction Using Standard Normalization, Stacking, and Blending

This notebook is prepared for your university project and performs the following steps:
1. Load and inspect the dataset final_cleaned_data.csv.
2. Apply Standard Normalization to:
   - Temperature
   - Moisture
   - Rainfall
   - PH
   - Nitrogen
   - Phosphorous
   - Potassium
   - Carbon
   - target_crop
3. Train and evaluate a Stacking model for target_fertilizer.
4. Train and evaluate a Blending model for target_fertilizer.
5. Compare final performance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

print('Libraries imported successfully.')

In [ ]:
# Step 1: Load dataset
df = pd.read_csv('final_cleaned_data.csv')

print('Dataset shape:', df.shape)
display(df.head())

In [ ]:
# Step 2: Check columns and missing values
print('Columns:\n', df.columns.tolist())

missing = df.isnull().sum()
print('\nMissing values per column:')
print(missing[missing > 0] if (missing > 0).any() else 'No missing values found.')

In [ ]:
# Step 3: Define target and normalization columns
target_col = 'target_fertilizer'

normalize_cols = [
    'Temperature', 'Moisture', 'Rainfall', 'PH',
    'Nitrogen', 'Phosphorous', 'Potassium', 'Carbon',
    'target_crop'
]

required_cols = normalize_cols + [target_col]
missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

print('Target column:', target_col)
print('Columns selected for standard normalization:', normalize_cols)

In [ ]:
# Step 4: Apply standard normalization
# We scale only the requested columns and keep other columns unchanged.
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[normalize_cols] = scaler.fit_transform(df_scaled[normalize_cols])

print('Normalized sample:')
display(df_scaled[normalize_cols].head())

In [ ]:
# Quick check of mean ~ 0 and std ~ 1 after scaling
stats = pd.DataFrame({
    'mean': df_scaled[normalize_cols].mean(),
    'std': df_scaled[normalize_cols].std()
})
display(stats)

In [ ]:
# Step 5: Prepare features and target for modeling
# float32 cuts memory usage roughly in half compared to float64.
X = df_scaled.drop(columns=[target_col]).astype(np.float32)
y = df_scaled[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('Unique classes in target_fertilizer:', y.nunique())

## Stacking Method

In [ ]:
# Step 6: Define base and meta models for stacking
# Memory-safe setup: keep multiple learners but avoid heavy parallel workers.
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=120, n_jobs=1, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=120, n_jobs=1, random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=9, weights='distance')),
    ('gnb', GaussianNB()),
    ('dt', DecisionTreeClassifier(max_depth=12, random_state=42))
]

meta_model = LogisticRegression(max_iter=2000, multi_class='auto')

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    stack_method='predict_proba',
    cv=3,
    n_jobs=1,
    passthrough=False
)

print(f'Stacking model initialized with {len(base_estimators)} base learners.')

In [ ]:
# Step 7: Train and evaluate stacking model
try:
    stack_model.fit(X_train, y_train)
except MemoryError:
    print('MemoryError detected. Retrying with a lighter stacking setup...')
    light_base_estimators = [
        ('rf', RandomForestClassifier(n_estimators=80, n_jobs=1, random_state=42)),
        ('et', ExtraTreesClassifier(n_estimators=80, n_jobs=1, random_state=42)),
        ('gb', GradientBoostingClassifier(random_state=42)),
        ('gnb', GaussianNB())
    ]
    stack_model = StackingClassifier(
        estimators=light_base_estimators,
        final_estimator=meta_model,
        stack_method='predict_proba',
        cv=3,
        n_jobs=1,
        passthrough=False
    )
    stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)
stack_acc = accuracy_score(y_test, stack_pred)
print(f'Stacking Accuracy: {stack_acc:.4f}')
print('\nStacking Classification Report:')
print(classification_report(y_test, stack_pred))

## Blending Method

In [ ]:
# Step 8: Create train-blend split from training data
X_base, X_blend, y_base, y_blend = train_test_split(
    X_train, y_train,
    test_size=0.3,
    random_state=42,
    stratify=y_train
)

print('Base training part :', X_base.shape)
print('Blending validation:', X_blend.shape)

In [ ]:
# Step 9: Train base models for blending
from sklearn.tree import DecisionTreeClassifier

blend_base_models = [
    RandomForestClassifier(n_estimators=120, n_jobs=1, random_state=42),
    ExtraTreesClassifier(n_estimators=120, n_jobs=1, random_state=42),
    GradientBoostingClassifier(random_state=42),
    KNeighborsClassifier(n_neighbors=9, weights='distance'),
    GaussianNB(),
    DecisionTreeClassifier(max_depth=12, random_state=42)
]

for model in blend_base_models:
    model.fit(X_base, y_base)

print(f'Base models trained for blending: {len(blend_base_models)}')

In [ ]:
# Step 10: Build meta-features using probability outputs
if 'blend_base_models' not in globals():
    print('blend_base_models not found. Re-running Step 9 setup...')
    from sklearn.tree import DecisionTreeClassifier
    blend_base_models = [
        RandomForestClassifier(n_estimators=120, n_jobs=1, random_state=42),
        ExtraTreesClassifier(n_estimators=120, n_jobs=1, random_state=42),
        GradientBoostingClassifier(random_state=42),
        KNeighborsClassifier(n_neighbors=9, weights='distance'),
        GaussianNB(),
        DecisionTreeClassifier(max_depth=12, random_state=42)
    ]
    for model in blend_base_models:
        model.fit(X_base, y_base)

blend_meta_train = np.column_stack([
    model.predict_proba(X_blend) for model in blend_base_models
]).astype(np.float32)

blend_meta_test = np.column_stack([
    model.predict_proba(X_test) for model in blend_base_models
]).astype(np.float32)

print('Meta-feature train shape:', blend_meta_train.shape)
print('Meta-feature test shape :', blend_meta_test.shape)

In [ ]:
# Step 11: Train meta-model and evaluate blending
blend_meta_model = LogisticRegression(max_iter=2000, multi_class='auto')
blend_meta_model.fit(blend_meta_train, y_blend)

blend_pred = blend_meta_model.predict(blend_meta_test)
blend_acc = accuracy_score(y_test, blend_pred)

print(f'Blending Accuracy: {blend_acc:.4f}')
print('\nBlending Classification Report:')
print(classification_report(y_test, blend_pred))

In [ ]:
# Step 11.1: Accuracy report and confusion matrix (Seaborn)
accuracy_report = pd.DataFrame({
    'Model': ['Stacking', 'Blending'],
    'Accuracy': [stack_acc, blend_acc]
}).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

print('Accuracy Report:')
display(accuracy_report)

stack_cm = confusion_matrix(y_test, stack_pred)
blend_cm = confusion_matrix(y_test, blend_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(stack_cm, cmap='Blues', annot=False, fmt='d', cbar=False, ax=axes[0])
axes[0].set_title('Stacking Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(blend_cm, cmap='Greens', annot=False, fmt='d', cbar=False, ax=axes[1])
axes[1].set_title('Blending Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

In [ ]:
# Step 12: Final comparison
comparison = pd.DataFrame({
    'Method': ['Stacking', 'Blending'],
    'Accuracy': [stack_acc, blend_acc]
}).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

display(comparison)

best_method = comparison.loc[0, 'Method']
best_score = comparison.loc[0, 'Accuracy']
print(f'Best method on this split: {best_method} (Accuracy = {best_score:.4f})')

## Conclusion
- Standard Normalization was applied to the requested columns, including target_crop.
- Both Stacking and Blending methods were trained for predicting target_fertilizer.
- Final comparison identifies the better method on your test split.

You can now use this notebook directly in your project report, including methodology and result tables.